## Annex F Python Script to make staad input file

creates staad .std input file from reading properties, fixities, and load .csv . also extracts deformations, support reactions, and member froces using openstaad python scripts. note that extracting results seem to default from kips, k-in, and inches for results

In [22]:
import pandas as pd

# 1. Load CSVs
props = pd.read_csv('properties.csv')
fixities = pd.read_csv('fixities.csv')
loads_slab = pd.read_csv('loads_slab.csv')
loads_sw = pd.read_csv('loads_sw.csv')

# 2. Combine and aggregate loads into a single data set
combined_loads = pd.concat([loads_sw, loads_slab], ignore_index=True)
aggregated_loads = combined_loads.groupby('node', as_index=False)[['Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz']].sum()

# 3. Generate the STAAD file
with open("MyModel.std", "w") as f:
    # --- HEADER & GLOBAL WORKSPACE ---
    f.write("STAAD SPACE\n")
    f.write("UNIT MM KN\n")  # Keeps properties (mm, kN/mm2) and inputs (kN, kN-mm) 1:1
    
    # --- GEOMETRY (JOINT COORDINATES) ---
    f.write("JOINT COORDINATES\n")
    nodes_i = props[['Node_I', 'Xi', 'Yi', 'Zi']].rename(columns={'Node_I': 'Node', 'Xi': 'X', 'Yi': 'Y', 'Zi': 'Z'})
    nodes_j = props[['Node_J', 'Xj', 'Yj', 'Zj']].rename(columns={'Node_J': 'Node', 'Xj': 'X', 'Yj': 'Y', 'Zj': 'Z'})
    unique_nodes = pd.concat([nodes_i, nodes_j]).drop_duplicates(subset=['Node']).sort_values('Node')
    
    for _, row in unique_nodes.iterrows():
        f.write(f"{int(row['Node'])} {row['X']:.3f} {row['Y']:.3f} {row['Z']:.3f}\n")
        
    # --- TOPOLOGY (MEMBER INCIDENCES) ---
    f.write("MEMBER INCIDENCES\n")
    for _, row in props.iterrows():
        f.write(f"{int(row['Member_ID'])} {int(row['Node_I'])} {int(row['Node_J'])}\n")
        
    # --- MEMBER SECTION PROPERTIES ---
    f.write("MEMBER PROPERTIES\n")  
    for _, row in props.iterrows():
        member = int(row['Member_ID'])
        ax = float(row['Area (mm2)'])
        ix = float(row['J (mm4)'])    
        iy = float(row['Iy (mm4)'])   
        iz = float(row['Iz (mm4)'])   
        f.write(f"{member} PRISMATIC AX {ax:.3f} IX {ix:.3f} IY {iy:.3f} IZ {iz:.3f}\n")
    
    # --- MATERIAL CONSTANTS ---
    f.write("CONSTANTS\n")
    for _, row in props.iterrows():
        mem_id = int(row['Member_ID'])
        e_kn_mm2 = row['E (GPa)']  # 1 GPa = 1 kN/mm2
        f.write(f"E {e_kn_mm2:.3f} MEMB {mem_id}\n")
        f.write(f"POISSON {row['v']:.3f} MEMB {mem_id}\n")

    # --- SUPPORTS ---
    f.write("SUPPORTS\n")
    for _, row in fixities.iterrows():
        releases = []
        if not row['x']: releases.append("FX")
        if not row['y']: releases.append("FY")
        if not row['z']: releases.append("FZ")
        if not row['x-rot']: releases.append("MX")
        if not row['y-rot']: releases.append("MY")
        if not row['z-rot']: releases.append("MZ")
        
        node_id = int(row['node'])
        if len(releases) == 0:
            f.write(f"{node_id} FIXED\n")
        elif len(releases) == 6:
            continue 
        else:
            release_str = " ".join(releases)
            f.write(f"{node_id} FIXED BUT {release_str}\n")
        
    # --- COMBINED LOADING CASE ---
    f.write("LOAD 1 TOTAL DEAD LOAD\n")
    f.write("JOINT LOAD\n")
    for _, row in aggregated_loads.iterrows():
        # Passed directly as kN and kN-mm under the active UNIT MM KN environment
        f.write(f"{int(row['node'])} FX {row['Fx']:.3f} FY {row['Fy']:.3f} FZ {row['Fz']:.3f} MX {row['Mx']:.6f} MY {row['My']:.6f} MZ {row['Mz']:.6f}\n")

    # --- ANALYSIS & DATA EXTRACTION ---
    f.write("PERFORM ANALYSIS\n")
    
    # --- UPDATE: FORCE OUTPUT REPORTING TO NEWTONS AND MM ---
    # This instructs the text printing engine to convert internal results to N and N-mm
    f.write("UNIT MM KN\n") 
    
    # These will now print out in mm, N, and N-mm
    f.write("PRINT JOINT DISPLACEMENTS\n")
    f.write("PRINT SUPPORT REACTIONS\n")
    f.write("PRINT MEMBER FORCES\n")
    
    f.write("FINISH\n")

print("STAAD model file 'MyModel-test.std' successfully compiled with Newton-based post-analysis reporting.")

STAAD model file 'MyModel-test.std' successfully compiled with Newton-based post-analysis reporting.


In [ ]:
import os
import win32com.client
import win32com.client.gencache as gencache

# 1. Connect to the running instance of STAAD
try:
    raw_os = win32com.client.GetActiveObject("StaadPro.OpenSTAAD")
    
    # 2. Force early-binding to reveal the hidden Type Library attributes
    os = gencache.EnsureDispatch(raw_os)
    print("Successfully established early-bound connection!\n")
except Exception as e:
    print(f"Error connecting: {e}. Ensure STAAD.Pro is open with a model.")
    exit()

# 3. List all top-level sub-objects/modules available at the root level
print("--- MAIN OPENSTAAD SUB-MODULES ---")
attributes = [attr for attr in dir(os) if not attr.startswith("_")]

for attr in attributes:
    print(f" - os.{attr}")

print("\n" + "="*50 + "\n")

# 4. Deep-dive into the primary result modules to see what functions they contain
modules_to_inspect = ['Geometry', 'Output', 'Load', 'Support', 'Property']

for mod_name in modules_to_inspect:
    if hasattr(os, mod_name):
        sub_obj = getattr(os, mod_name)
        print(f"--- METHODS AVAILABLE INSIDE: os.{mod_name} ---")
        methods = [m for m in dir(sub_obj) if not m.startswith("_")]
        
        # Print first 15 methods as a sample (to keep your terminal readable)
        for m in methods[:15]:
            print(f"   .{m}")
        if len(methods) > 15:
            print(f"   ... and {len(methods) - 15} more functions.")
        print()

In [62]:
import pandas as pd
from openstaad import Output  # Assuming your modified class is imported here

# Initialize connection
output = Output()
load_case = 1

# Load your layout setup tracking files
props = pd.read_csv('properties.csv')
fixities = pd.read_csv('fixities.csv')

# Find all unique node IDs mentioned in your model properties
all_nodes = pd.concat([props['Node_I'], props['Node_J']]).unique()
support_nodes = fixities['node'].unique()
member_ids = props['Member_ID'].unique()

# # =====================================================================
# # 1. EXTRACT GLOBAL JOINT DISPLACEMENTS
# # =====================================================================
# print("Extracting global joint displacements...")
# disp_records = []
# for node in all_nodes:
#     node_id = int(node)
#     disp_data = output.GetNodeDisplacements(node=node_id, lc=load_case)
#     disp_records.append([node_id, *disp_data])

# df_disp = pd.DataFrame(disp_records, columns=['Node_ID', 'X', 'Y', 'Z', 'RX', 'RY', 'RZ'])
# df_disp.to_csv('outputs_displacements_clean.csv', index=False, float_format='%.6f')

# =====================================================================
# 2. EXTRACT SUPPORT REACTIONS
# =====================================================================
print("Extracting support reactions...")
reaction_records = []
for node in support_nodes:
    node_id = int(node)
    react_data = output.GetSupportReactions(node=node_id, lc=load_case)
    reaction_records.append([node_id, *react_data])

df_react = pd.DataFrame(reaction_records, columns=['Node_ID', 'Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz'])

# convert 1 kip = 4.4482216 kN 
force_column = ['Fx', 'Fy', 'Fz']
df_react[force_column] = df_react[force_column] * 4.4482216

# convert 1 kip-in = 112.985 kN-mm
moment_column = ['Mx', 'My', 'Mz']
df_react[moment_column] = df_react[moment_column] * 112.985

df_react.to_csv('outputs_reactions_clean.csv', index=False, float_format='%.12f')

# =====================================================================
# 3. EXTRACT MEMBER END FORCES
# =====================================================================
print("Extracting member end forces...")
force_records = []
for member in member_ids:
    member_id = int(member)
    forces_i = output.GetMemberEndForces(beam=member_id, start=True, lc=load_case)
    forces_j = output.GetMemberEndForces(beam=member_id, start=False, lc=load_case)
    
    force_records.append([member_id, 'Node_I', *forces_i])
    force_records.append([member_id, 'Node_J', *forces_j])

df_forces = pd.DataFrame(force_records, columns=['Member_ID', 'End', 'Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz'])

# convert 1 kip = 4.4482216 kN 
force_column = ['Fx', 'Fy', 'Fz']
df_forces[force_column] = df_forces[force_column] * 4.4482216

# convert 1 kip-in = 112.985 kN-mm
moment_column = ['Mx', 'My', 'Mz']
df_forces[moment_column] = df_forces[moment_column] * 112.985

df_forces.to_csv('outputs_member_forces_clean.csv', index=False, float_format='%.12f')

print("\nAll data successfully exported to clean CSV files!")

Extracting support reactions...
Extracting member end forces...

All data successfully exported to clean CSV files!


In [60]:
df_forces

,Member_ID,End,Fx,Fy,Fz,Mx,My,Mz
0,1,Node_I,112.385767,-0.040970,0.053677,66.666272,432.417747,1781.497444
1,1,Node_J,-112.385767,0.040970,-0.053677,-66.666272,-512.933039,-1842.952735
2,2,Node_I,157.366546,-2.161353,-0.896094,6.657307,1829.981898,-3039.356582
3,2,Node_J,-157.366546,2.161353,0.896094,-6.657307,-485.838108,-202.678432
4,3,Node_I,99.872403,-2.290690,-0.703987,-39.205310,1551.393365,-4900.912293
...,...,...,...,...,...,...,...,...
379,190,Node_J,1.960696,-20.411435,-4.290900,1153.137123,-2963.366558,5399.415332
380,191,Node_I,-4.079664,5.972263,2.838900,4662.061751,-1584.304583,-1736.496469
381,191,Node_J,4.079664,-5.972263,-2.838900,-4662.061751,-3596.695919,12635.894597
382,192,Node_I,0.912187,-2.443943,-2.118967,-5011.041894,3687.710081,-6442.590083


In [12]:
from comtypes import automation
from comtypes import client
import ctypes

# Connect to OpenSTAAD
os = client.GetActiveObject("StaadPro.OpenSTAAD")
geometry = os.Geometry
output = os.Output

# Flag the method as callable
os._FlagAsMethod("GetInputUnitForLength")

# Call the method (no arguments needed)
unit_code = os.GetInputUnitForLength()
print("Displacement unit code:", unit_code)

# Helper: create a safe array of doubles
def make_safe_array_double(size):
    return automation._midlSAFEARRAY(ctypes.c_double).create([0]*size)

# Helper: wrap array in a variant reference
def make_variant_vt_ref(obj, var_type):
    var = automation.VARIANT()
    var._.c_void_p = ctypes.addressof(obj)
    var.vt = var_type | automation.VT_BYREF
    return var

# Example: Get global displacements for node 1, load case 1
nodeNo = 5
loadcaseNo = 1

# 6 values: UX, UY, UZ, RX, RY, RZ
safe_array_disp = make_safe_array_double(6)
displacements = make_variant_vt_ref(safe_array_disp, automation.VT_ARRAY | automation.VT_R8)

# Ensure method is flagged as callable
output._FlagAsMethod("GetNodeDisplacements")

# Call the method
output.GetNodeDisplacements(nodeNo, loadcaseNo, displacements)

# Print the results
print("Node Displacements (UX, UY, UZ, RX, RY, RZ):")
for value in safe_array_disp:
    print(value)

Displacement unit code: 5
Node Displacements (UX, UY, UZ, RX, RY, RZ):
(-0.001936799963004887, -0.004322465043514967, 0.00854059774428606, 0.00024235289311036468, -4.996817324354197e-07, 4.749788422486745e-05)


In [4]:
from comtypes import automation
from comtypes import client
import ctypes

# Connect to OpenSTAAD
os = client.GetActiveObject("StaadPro.OpenSTAAD")
geometry = os.Geometry
output = os.Output

# Helper: create a safe array of doubles
def make_safe_array_double(size):
    return automation._midlSAFEARRAY(ctypes.c_double).create([0]*size)

# Helper: wrap array in a variant reference
def make_variant_vt_ref(obj, var_type):
    var = automation.VARIANT()
    var._.c_void_p = ctypes.addressof(obj)
    var.vt = var_type | automation.VT_BYREF
    return var

for i in range(111):
    print(i)
    # Example: Get global displacements for node 1, load case 1
    nodeNo = i+1
    loadcaseNo = 1

    # 6 values: UX, UY, UZ, RX, RY, RZ
    safe_array_disp = make_safe_array_double(6)
    displacements = make_variant_vt_ref(safe_array_disp, automation.VT_ARRAY | automation.VT_R8)

    # Ensure method is flagged as callable
    output._FlagAsMethod("GetNodeDisplacements")

    # Call the method
    output.GetNodeDisplacements(nodeNo, loadcaseNo, displacements)

    # Print the results
    print("Node Displacements (UX, UY, UZ, RX, RY, RZ):")
    for value in safe_array_disp:
        print(value)

0
Node Displacements (UX, UY, UZ, RX, RY, RZ):
(0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
1
Node Displacements (UX, UY, UZ, RX, RY, RZ):
(0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
2
Node Displacements (UX, UY, UZ, RX, RY, RZ):
(0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
3
Node Displacements (UX, UY, UZ, RX, RY, RZ):
(0.0017818514024838805, -0.00308695575222373, 0.0028382870368659496, 9.893195965560153e-05, -5.0039661800838076e-06, -6.06883731961716e-05)
4
Node Displacements (UX, UY, UZ, RX, RY, RZ):
(-0.001936799963004887, -0.004322465043514967, 0.00854059774428606, 0.00024235289311036468, -4.996817324354197e-07, 4.749788422486745e-05)
5
Node Displacements (UX, UY, UZ, RX, RY, RZ):
(-0.0037136103492230177, -0.002743244869634509, 0.007412496954202652, 0.00021420004486571997, 2.9427762910927413e-06, 0.00010658972314558923)
6
Node Displacements (UX, UY, UZ, RX, RY, RZ):
(0.01566881127655506, -0.008685467764735222, 0.02718108519911766, 0.00033798479125835, -1.4678301340609323e-05, -0.00018385493604000658)
7
Node Displacements

In [15]:
import pandas as pd
from comtypes import automation
from comtypes import client
import ctypes

# 1. Connect to OpenSTAAD
try:
    os = client.GetActiveObject("StaadPro.OpenSTAAD")
    output = os.Output
except Exception as e:
    print(f"Failed to connect to active STAAD instance: {e}")
    exit()

# Helper: create a safe array of doubles
def make_safe_array_double(size):
    return automation._midlSAFEARRAY(ctypes.c_double).create([0]*size)

# Helper: wrap array in a variant reference
def make_variant_vt_ref(obj, var_type):
    var = automation.VARIANT()
    var._.c_void_p = ctypes.addressof(obj)
    var.vt = var_type | automation.VT_BYREF
    return var

# Ensure method is flagged as callable
output._FlagAsMethod("GetNodeDisplacements")

loadcaseNo = 1
records = []

print("Extracting node displacements...")

# 2. Loop through nodes 1 to 111
for i in range(111):
    nodeNo = i + 1

    # Initialize a clean safe array for 6 values
    safe_array_disp = make_safe_array_double(6)
    displacements = make_variant_vt_ref(safe_array_disp, automation.VT_ARRAY | automation.VT_R8)

    # Invoke the OpenSTAAD method
    output.GetNodeDisplacements(nodeNo, loadcaseNo, displacements)

    # --- FIX: Grab the full unpacked tuple from index 0, then unpack it ---
    ux, uy, uz, rx, ry, rz = safe_array_disp[0]

    # Append as a completely flat 7-item list row
    records.append([nodeNo, ux, uy, uz, rx, ry, rz])

# 3. Create the DataFrame (7 columns matching 7 items perfectly)
columns = ['Node_ID', 'UX', 'UY', 'UZ', 'RX', 'RY', 'RZ']
df_disp = pd.DataFrame(records, columns=columns)

# --- NEW: Convert translation columns from inches to millimeters ---
# 1 inch = 25.4 mm
linear_conversion_columns = ['UX', 'UY', 'UZ']
df_disp[linear_conversion_columns] = df_disp[linear_conversion_columns] * 25.4

# 4. Save to CSV with high-precision decimals
df_disp.to_csv('outputs_displacements.csv', index=False, float_format='%.12f')

print("\nExtraction & Unit Conversion Complete! Dataframe preview (Translations in mm):")
print(df_disp.head(10))

Extracting node displacements...

Extraction & Unit Conversion Complete! Dataframe preview (Translations in mm):
   Node_ID        UX        UY        UZ        RX            RY        RZ
0        1  0.000000  0.000000  0.000000  0.000000  0.000000e+00  0.000000
1        2  0.000000  0.000000  0.000000  0.000000  0.000000e+00  0.000000
2        3  0.000000  0.000000  0.000000  0.000000  0.000000e+00  0.000000
3        4  0.045259 -0.078409  0.072092  0.000099 -5.003966e-06 -0.000061
4        5 -0.049195 -0.109791  0.216931  0.000242 -4.996817e-07  0.000047
5        6 -0.094326 -0.069678  0.188277  0.000214  2.942776e-06  0.000107
6        7  0.397988 -0.220611  0.690400  0.000338 -1.467830e-05 -0.000184
7        8  0.028231 -0.312665  0.696564 -0.000087 -1.465733e-06 -0.000169
8        9 -0.333077 -0.195002  0.700845  0.000002  8.632143e-06 -0.000014
9       10  0.701114 -0.317624  1.050816  0.000277 -6.893867e-05 -0.000116


In [3]:
import os
import nbformat
from nbconvert import HTMLExporter
from weasyprint import HTML, CSS

def export_notebook_to_annex_F(notebook_path, output_pdf_path="Annex_F_Report.pdf"):
    print(f"Reading notebook: {notebook_path}...")
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook_content = nbformat.read(f, as_version=4)
        
    # 1. Initialize the HTML Exporter from nbconvert
    html_exporter = HTMLExporter()
    # Optional: exclude input or output prompts if you want a cleaner look
    # html_exporter.exclude_input_prompt = True
    # html_exporter.exclude_output_prompt = True
    
    # 2. Convert the notebook to standard HTML string
    (body, resources) = html_exporter.from_notebook_node(notebook_content)
    
    # 3. Define the strict CSS Paged Media rules for Annex F
    # This automatically builds the header, footer page counts, and handles line wrapping
    custom_css = """
    @page {
        size: A4 portrait;
        margin: 20mm 15mm 20mm 15mm;
        
        @top-right {
            content: "Annex F. Python-Staad input and extract results code";
            font-family: 'Courier New', Courier, monospace;
            font-size: 8.5pt;
            color: #333333;
            font-weight: bold;
        }
        
        @bottom-right {
            /* WeasyPrint dynamically computes page and pages variables natively */
            content: "Page F." counter(page) " of F." counter(pages);
            font-family: 'Courier New', Courier, monospace;
            font-size: 8.5pt;
            color: #333333;
        }
    }
    
    /* Global formatting fixes for high-quality printing */
    body {
        font-family: 'Courier New', Courier, monospace !important;
        font-size: 9pt !important;
    }
    
    /* Force long lines of code and text inputs to wrap cleanly instead of clipping horizontally */
    pre, code, .highlight, .input_area, .output_text pre {
        white-space: pre-wrap !important;
        word-wrap: break-word !important;
        word-break: break-all !important;
    }
    
    /* Ensure markdown headers inside the notebook look clean and intentional */
    h1, h2, h3, h4 {
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif !important;
        page-break-after: avoid !important;
    }
    
    /* Avoid cutting an atomic code cell cleanly across two pages if possible */
    .cell {
        page-break-inside: auto !important;
    }
    .input_area, .output_wrapper {
        page-break-inside: avoid !important;
    }
    """
    
    # 4. Inject our custom CSS block directly into the HTML body header
    styled_html = body.replace("</head>", f"<style>{custom_css}</style></head>")
    
    print("Compiling styled HTML into final PDF...")
    # 5. Compile to PDF via WeasyPrint
    HTML(string=styled_html).write_pdf(output_pdf_path)
    print(f"Successfully generated: {output_pdf_path}")

# ==============================================================================
# EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # Replace with your actual notebook file name
    target_notebook = "staad.ipynb" 
    
    if os.path.exists(target_notebook):
        export_notebook_to_annex_F(target_notebook, "Annex_F_PythonStaad_Code.pdf")
    else:
        print(f"Error: Could not find notebook file '{target_notebook}' in the current directory.")

Reading notebook: staad.ipynb...
Compiling styled HTML into final PDF...
Successfully generated: Annex_F_PythonStaad_Code.pdf
